In [1]:
# src/process_global

# This file is responsible for finding the minmax params of all cities for the generalization and GNN preprocessing pipeline.
# It reads all the csvs and finds the minmax params for all cities combined.
# This is a pre-processing step before the main pipeline is run.

from pipeline.stage import (
    calculate_normalization_params,
    normalize_dataset,
    prepare_io_data,
    export_combined_data,
)
from pipeline.extract import read_all_city_train_frames
import pandas as pd


Running __init__.py for data pipeline...
Pipeline initialized



In [2]:
city_train_frames = read_all_city_train_frames(
    "/home/nick/bachelor-project/forecasting_smog_DL_GNN/data/data_combined/all"
)


In [ ]:
city_train_frames

In [4]:
# Find global min/max values for each pollutant across all cities and years
global_min_max = {}

# Loop through each city dictionary
for city_dict in city_train_frames:
    # Loop through each year in the 'train' key
    for year, pollutant_dict in city_dict['train'].items():
        # Loop through each pollutant dataframe
        for pollutant, df in pollutant_dict.items():
            if pollutant not in global_min_max:
                global_min_max[pollutant] = {'min': float('inf'), 'max': float('-inf')}
            
            # Update min/max if current df has more extreme values
            curr_min = df.min().min()  # Get minimum value across all columns
            curr_max = df.max().max()  # Get maximum value across all columns
            
            if curr_min < global_min_max[pollutant]['min']:
                global_min_max[pollutant]['min'] = curr_min
            
            if curr_max > global_min_max[pollutant]['max']:
                global_min_max[pollutant]['max'] = curr_max

# Now global_min_max contains the min and max values for each pollutant across all cities and years
global_min_max

{'O3': {'min': -5.0, 'max': 205.0},
 'NO2': {'min': -2.6, 'max': 177.6},
 'temp': {'min': -44.0, 'max': 339.0},
 'dewP': {'min': -62.0, 'max': 223.0},
 'WD': {'min': 0.0, 'max': 360.0},
 'Wvh': {'min': 0.0, 'max': 170.0},
 'Wmax': {'min': 0.0, 'max': 260.0},
 'preT': {'min': 0.0, 'max': 10.0},
 'P': {'min': 9681.0, 'max': 10396.0},
 'preS': {'min': -1.0, 'max': 326.0},
 'SQ': {'min': 0.0, 'max': 10.0},
 'Q': {'min': 0.0, 'max': 316.0}}

In [5]:
meteo_vars = {
    "temp": {"code": "T"},
    "dewP": {"code": "TD"},
    "WD": {"code": "DD"},
    "Wvh": {"code": "FH"},
    "Wmax": {"code": "FX"},
    "preT": {"code": "DR"},
    "P": {"code": "P"},
    "preS": {"code": "RH"},
    "SQ": {"code": "SQ"},
    "Q": {"code": "Q"},
}

years = [2017, 2018, 2020, 2021, 2022, 2023]

contaminants = ['NO2', 'O3']

In [ ]:
norms = []
for city_frame in city_train_frames:
    
    norm = normalize_dataset(city_frame, global_min_max, years=years, contaminants=contaminants, meteo_vars=meteo_vars)
    norms.append(norm)
len(norms)

sensors = [
    ["NL01485", "NL01494"],   # Rotterdam
    ["NL10636", "NL10641"],   # Utrecht
    ["NL49003", "NL49012"],   # Amsterdam
    ]

meteo_target = ["temp", "dewP", "WD", "Wvh", "P", "SQ"]

In [ ]:
io_frames = []

for idx, io_frame in enumerate(norms):
    io_frames.append(prepare_io_data(io_frame, years, ["train", "val", "test"], sensors[idx], contaminants, meteo_vars))
io_frames

[{'u': {'train': {2017: [                      NL01485
     DateTime                     
     2017-08-01 00:00:00  0.379023
     2017-08-01 01:00:00  0.345172
     2017-08-01 02:00:00  0.310211
     2017-08-01 03:00:00  0.268590
     2017-08-01 04:00:00  0.187014
     ...                       ...
     2017-12-30 19:00:00  0.066593
     2017-12-30 20:00:00  0.077137
     2017-12-30 21:00:00  0.066593
     2017-12-30 22:00:00  0.063263
     2017-12-30 23:00:00  0.062153
     
     [3648 rows x 1 columns],
                           NL01485
     DateTime                     
     2017-08-01 00:00:00  0.027143
     2017-08-01 01:00:00  0.025714
     2017-08-01 02:00:00  0.028095
     2017-08-01 03:00:00  0.070952
     2017-08-01 04:00:00  0.119048
     ...                       ...
     2017-12-30 19:00:00  0.316667
     2017-12-30 20:00:00  0.307619
     2017-12-30 21:00:00  0.313333
     2017-12-30 22:00:00  0.326667
     2017-12-30 23:00:00  0.315238
     
     [3648 rows x 1 columns]

In [17]:
# Export io frames

for idx, frame in enumerate(io_frames):
    export_combined_data(frame,
        output_dir=f"../data/data_combined/all/{idx}",
        contaminants=contaminants,
        meteo_target=meteo_target)